> ⚠️ **机器翻译提示**：本 notebook 由机器翻译生成，未经人工校对，如发现翻译错误敬请谅解（代码与计算结果和越南语原版完全一致，只有文字说明是机器翻译）。

# 第07单元 — 卡方检验与非参数检验
第07单元专属 notebook —— 重现课程中的全部6项检验：卡方检验（2个表）、Mann-Whitney U检验、Wilcoxon检验、Kruskal-Wallis检验、Friedman检验。

SPSS 对应操作：`Analyze > Descriptive Statistics > Crosstabs`（卡方）、`Analyze > Nonparametric Tests > Legacy Dialogs`。

In [ ]:
# 先运行这一格（Colab：点击 ▶）。无需安装任何库。
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from scipy import stats
sns.set_theme(style="whitegrid"); plt.rcParams["figure.figsize"] = (7, 4)
pd.set_option("display.precision", 3)

def make_data(seed=2026, n=240):
    """模拟数据集"240人班级" —— 第03-08单元统一使用 seed=2026，
    与仓库中的 lop_hoc_240.csv/.sav 完全一致（非真实数据，但100%可复现）。"""
    rng = np.random.default_rng(seed)
    school = rng.choice(list("ABC"), n, p=[.35, .35, .30])
    gender = rng.choice(["男", "女"], n)
    method = rng.choice(["传统教学", "项目式教学"], n)
    study_hours = np.clip(rng.gamma(4, 1.2, n), 0.5, 15).round(1)
    interest = rng.normal(0, 1, n) + 0.15 * (method == "项目式教学")
    anxiety = rng.normal(0, 1, n) - 0.25 * interest
    def likert(lat, load, noise=0.7):
        return np.clip(np.round(3 + load * lat + rng.normal(0, noise, n)), 1, 5).astype(int)
    d = pd.DataFrame({"id": np.arange(1, n + 1), "school": school, "gender": gender, "method": method,
                      "study_hours": study_hours})
    for k, (lat, load) in enumerate([(interest, .8), (interest, .7), (interest, .75)], 1):
        d[f"h{k}"] = likert(lat, load)
    for k, (lat, load) in enumerate([(anxiety, .8), (anxiety, .75), (anxiety, .7)], 1):
        d[f"a{k}"] = likert(lat, load)
    eff = d.school.map({"A": 3, "B": 0, "C": -3}).to_numpy()
    d["pretest"] = (rng.normal(60, 10, n) + eff).round(1)
    d["posttest"] = np.clip(d.pretest + 3 + 5 * (method == "项目式教学") + rng.normal(0, 6, n), 0, 100).round(1)
    d["math"] = np.clip(35 + 2.5 * study_hours + 4 * interest - 3 * anxiety + eff + rng.normal(0, 6, n), 0, 100).round(1)
    d.loc[6, "study_hours"] = 48.0
    d.loc[[11, 57, 130], "h2"] = np.nan
    d["h2"] = d["h2"].astype("Int64")
    return d

df = make_data()
print("已生成 df：", df.shape)

## 1. 卡方检验：性别 × 教学方法（2×2表，含耶茨连续性校正）

🎯 **这个方法用来回答什么问题？** 卡方检验用于检验：两个定类变量是否彼此独立——换句话说，知道其中一个变量的取值，是否能比随机猜测更好地预测另一个变量的取值。原假设H0：两个变量相互独立，即交叉表每个单元格的观测频数与「两者毫无关联」情况下的期望频数相符。

In [ ]:
ct1 = pd.crosstab(df["gender"], df["method"])
display(ct1)
chi2, p, dof, exp = stats.chi2_contingency(ct1, correction=True)
print(f"卡方({dof})={chi2:.2f}, p={p:.3f}（2x2表已作耶茨校正）")

#### 📤 实际输出
χ²(1)=1.37, p=0.243 → 不显著。✅ 与课程一致。

## 2. 卡方检验：性别 × 学校（2×3表）+ Cramer's V

🎯 **这个方法用来回答什么问题？** Cramer's V 衡量两个定类变量之间关联的强度（作用类似于Pearson相关中的|r|）——取值范围从0（无关联）到1（完全关联）。与p值（只说明是否具有统计显著性）不同，Cramer's V 说明这种关联究竟有多强——这是两个不同的问题，呼应了第04单元的教训（统计显著性≠实际大小）。

In [ ]:
ct2 = pd.crosstab(df["gender"], df["school"])
display(ct2)
chi2b, pb, dofb, expb = stats.chi2_contingency(ct2, correction=False)
n_total = ct2.values.sum()
cramer_v = np.sqrt(chi2b / (n_total * (min(ct2.shape)-1)))
pct_small = (expb < 5).mean() * 100
print(f"卡方({dofb})={chi2b:.2f}, p={pb:.3f}, Cramer's V={cramer_v:.3f}")
print(f"期望频数<5的单元格占比：{pct_small:.0f}%（安全规则：应<20%）")

#### 📤 实际输出
χ²(2)=0.14, p=0.932, Cramer's V≈0.024（接近0），0%的单元格违反20%规则 → 结果可信，不显著。

## 3. Mann-Whitney U检验：`h1` 评分按性别分组

🎯 **这个方法用来回答什么问题？** Mann-Whitney U检验是独立样本t检验的非参数版本：检验两组中是否有一组的取值倾向于高于另一组——依据的是将两组合并后对全部观测值的排名，而不是均值。适用于顺序数据（如李克特量表）或违反t检验正态性假设的情况。

In [ ]:
h1_male = df.loc[df["gender"]=="男", "h1"].dropna()
h1_female = df.loc[df["gender"]=="女", "h1"].dropna()
u, p = stats.mannwhitneyu(h1_male, h1_female)
print(f"U={u:.1f}, p={p:.3f}")

#### 📤 实际输出
U=6416, p=0.127 → 不显著。

## 4. Wilcoxon检验：比较两项评分 `a1` 与 `a2`（同一240名学生）

🎯 **这个方法用来回答什么问题？** Wilcoxon检验是配对样本t检验的非参数版本：检验同一对象两次测量之间的差值是否倾向于偏向某一方向（正或负）——依据的是差值大小的排名，而不是差值的平均值。

In [ ]:
w_stat, p_w = stats.wilcoxon(df["a1"], df["a2"])
print(f"W={w_stat:.1f}, p={p_w:.3f}")
print(f"a1平均值={df['a1'].mean():.3f}, a2平均值={df['a2'].mean():.3f}")

#### 📤 实际输出
p=0.920 —— 完全没有差异（a1平均值≈2.908，a2平均值≈2.904，非常接近）。

## 5. Kruskal-Wallis检验：`h1` 评分按3所学校分组

🎯 **这个方法用来回答什么问题？** Kruskal-Wallis检验是单因素方差分析的非参数版本：检验3个或以上独立组的排名分布是否存在差异——适用于顺序数据或违反方差分析正态性假设的情况。和方差分析一样，显著结果并不能说明具体是哪一组存在差异。

In [ ]:
groups_h1 = [g["h1"].dropna().values for _, g in df.groupby("school")]
h_stat, p_kw = stats.kruskal(*groups_h1)
print(f"H={h_stat:.3f}, p={p_kw:.3f}")

#### 📤 实际输出
H=2.844, p=0.241 —— 不显著。与第06单元按学校比较数学成绩的显著结果（F=5.87, p=0.003）不同 —— 不同变量，不同结论。

## 6. Friedman检验：3项评分 `a1`、`a2`、`a3`（同一240名学生）

🎯 **这个方法用来回答什么问题？** Friedman检验是重复测量方差分析的非参数版本：检验同一组研究对象的3个或以上测量/变量，在排名上是否存在差异（基于每个个体内部的排名，而非原始平均值）。

In [ ]:
chi2_f, p_f = stats.friedmanchisquare(df["a1"], df["a2"], df["a3"])
print(f"卡方={chi2_f:.3f}, p={p_f:.3f}")

#### 📤 实际输出
χ²=0.555, p=0.758 —— 不显著，学生对这些相关方面的评价相当一致。